# 06 — M2 Verbal Harassment Corpus V2

**Goal:** Build a CLEAN verbal harassment corpus for M2 four-class router.

**V2 = MAJOR OVERHAUL.** The V1 corpus was ~70% noisy (Davidson 65% + Kaggle 25%).

**Changes:**
1. `normalize_text()` on ALL text
2. **Davidson REMOVED entirely** (46% annotator agreement, double-dipping with discrimination)
3. **HateXplain FIXED** (correct JSON parsing → majority-vote `label='offensive'`)
4. **OLID IND ADDED** (`subtask_c='IND'` = individual-targeted insults)
5. **HASOC OFFN ADDED** (English tweets labeled offensive)
6. **Kaggle CLEANED** (insult keyword filter removes ~30% false positives)
7. **TRAC-3 verbal REMOVED** (Bengali/Hindi)
8. TweetEval Offensive kept (uncapped)
9. ConvAbuse verbal kept
10. Cap ~7,000 for class balance

In [ ]:
# ============================================================
# CELL 1 — Setup + Shared Normalizer
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import pandas as pd
import numpy as np
import re
import json
import hashlib
from collections import Counter

BASE      = Path("/content/drive/MyDrive/AEGIS_ML")
RAW       = BASE / "data" / "raw"
THREAT    = RAW / "threat_gap"
PROCESSED = BASE / "data" / "processed" / "v5"
PROCESSED.mkdir(parents=True, exist_ok=True)

def normalize_text(text: str) -> str:
    """Canonical text normalization for AEGIS pipeline."""
    text = str(text).strip()
    text = text.lower()
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    text = re.sub(r'@\w+', '', text)
    text = re.sub(r'#(\w+)', r'\1', text)
    text = re.sub(r'\brt\s+', '', text)
    text = re.sub(r'&amp;', '&', text)
    text = re.sub(r'&lt;', '<', text)
    text = re.sub(r'&gt;', '>', text)
    text = re.sub(r'&#\d+;', '', text)
    text = re.sub(r'<user>', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text if text else None

def text_hash(t):
    return hashlib.sha256(t.strip().encode('utf-8')).hexdigest()

print("✅ Setup complete")

In [ ]:
# ============================================================
# CELL 2 — Load All Verbal Harassment Sources
# ============================================================

frames = []
source_tracker = {}  # track rows per source

print("="*70)
print("VERBAL HARASSMENT CORPUS V2 — LOADING SOURCES")
print("="*70)

# ══════════════════════════════════════════════════════════════
# ❌ DAVIDSON REMOVED — 46% annotator agreement, double-dips
# with discrimination. Was 65% of V1 corpus = dominant noise.
# ══════════════════════════════════════════════════════════════
print("\n❌ [REMOVED] Davidson — too noisy (46% agreement), double-dips")

# ── 1. TweetEval Offensive (label=1) ─────────────────────────
print("\n[1] TweetEval Offensive (label=1)")
df = pd.read_csv(RAW / 'tweeteval_offensive.csv')
df = df[df['label'] == 1].copy()
df['text'] = df['text'].apply(normalize_text)
df = df[df['text'].notna() & (df['text'].str.len() > 0)]
df['label'] = 'verbal_harassment'
frames.append(df[['text', 'label']].copy())
source_tracker['tweeteval_offensive'] = len(df)
print(f"  ✅ {len(df):,} rows")

# ── 2. HateXplain OFFENSIVE (FIXED PARSER) ──────────────────
print("\n[2] HateXplain (FIXED — majority-vote offensive)")
with open(RAW / 'hatexplain.json') as f:
    hx_raw = json.load(f)

hx_rows = []
for post_id, entry in hx_raw.items():
    tokens = entry.get('post_tokens', [])
    text = ' '.join(tokens)
    # Majority vote across annotators
    annotator_labels = [a['label'] for a in entry.get('annotators', [])]
    if not annotator_labels:
        continue
    majority = Counter(annotator_labels).most_common(1)[0][0]
    hx_rows.append({'text': text, 'hx_label': majority})

df_hx = pd.DataFrame(hx_rows)
# Only take 'offensive' — NOT 'hatespeech' (goes to discrimination)
df_hx_off = df_hx[df_hx['hx_label'] == 'offensive'].copy()
df_hx_off['text'] = df_hx_off['text'].apply(normalize_text)
df_hx_off = df_hx_off[df_hx_off['text'].notna() & (df_hx_off['text'].str.len() > 0)]
df_hx_off['label'] = 'verbal_harassment'
frames.append(df_hx_off[['text', 'label']].copy())
source_tracker['hatexplain_offensive'] = len(df_hx_off)
print(f"  ✅ {len(df_hx_off):,} rows (from {len(df_hx)} total parsed entries)")
print(f"  📊 HateXplain label distribution: {df_hx['hx_label'].value_counts().to_dict()}")

# ── 3. OLID IND (individual-targeted insults) ────────────────
print("\n[3] OLID IND (subtask_a=OFF + subtask_c=IND)")
try:
    df = pd.read_csv(RAW / 'olid' / 'olid_full.tsv', sep='\t')
    df = df[(df['subtask_a'] == 'OFF') & (df['subtask_c'] == 'IND')].copy()
    df['text'] = df['tweet'].apply(normalize_text)
    df = df[df['text'].notna() & (df['text'].str.len() > 0)]
    df['label'] = 'verbal_harassment'
    frames.append(df[['text', 'label']].copy())
    source_tracker['olid_ind'] = len(df)
    print(f"  ✅ {len(df):,} rows")
except Exception as e:
    print(f"  ⚠️ OLID loading failed: {e}")
    print("  Trying alternative: olid_routed.csv with IND filter...")
    try:
        df = pd.read_csv(THREAT / 'olid_routed.csv')
        df = df[df['label'] == 'verbal_harassment'].copy()
        df['text'] = df['text'].apply(normalize_text)
        df = df[df['text'].notna() & (df['text'].str.len() > 0)]
        df['label'] = 'verbal_harassment'
        frames.append(df[['text', 'label']].copy())
        source_tracker['olid_routed_vh'] = len(df)
        print(f"  ✅ {len(df):,} rows (from olid_routed.csv)")
    except Exception as e2:
        print(f"  ⚠️ Fallback also failed: {e2}")

# ── 4. HASOC OFFN (English tweets — offensive) ───────────────
print("\n[4] HASOC OFFN (task_2/task2 = OFFN)")
try:
    # Load 2019
    h19_train = pd.read_csv(RAW / 'hosoc_english/english_dataset_2019/english_dataset.tsv', sep='\t')
    h19_test  = pd.read_csv(RAW / 'hosoc_english/english_dataset_2019/hasoc2019_en_test-2919.tsv', sep='\t')
    h19 = pd.concat([h19_train, h19_test], ignore_index=True)
    h19 = h19.rename(columns={'task_2': 'task2'})
    h19_offn = h19[h19['task2'] == 'OFFN'][['text']].copy()

    # Load 2020
    h20_train = pd.read_excel(RAW / 'hosoc_english/english_dataset_2020/hasoc_2020_en_train_new.xlsx')
    h20_test  = pd.read_excel(RAW / 'hosoc_english/english_dataset_2020/hasoc_2020_en_test_new.xlsx')
    h20 = pd.concat([h20_train, h20_test], ignore_index=True)
    h20_offn = h20[h20['task2'] == 'OFFN'][['text']].copy()

    df_hasoc_offn = pd.concat([h19_offn, h20_offn], ignore_index=True)
    df_hasoc_offn['text'] = df_hasoc_offn['text'].apply(normalize_text)
    df_hasoc_offn = df_hasoc_offn[df_hasoc_offn['text'].notna() & (df_hasoc_offn['text'].str.len() > 0)]
    df_hasoc_offn['label'] = 'verbal_harassment'
    frames.append(df_hasoc_offn[['text', 'label']].copy())
    source_tracker['hasoc_offn'] = len(df_hasoc_offn)
    print(f"  ✅ {len(df_hasoc_offn):,} rows")
except Exception as e:
    print(f"  ⚠️ HASOC OFFN loading failed: {e}")

# ── 5. Kaggle Cyberbullying (keyword-filtered) ───────────────
print("\n[5] Kaggle Cyberbullying (other_cyberbullying + insult keywords)")
df = pd.read_csv(RAW / 'cyberbullying' / 'cyberbullying_tweets.csv')
df = df[df['cyberbullying_type'] == 'other_cyberbullying'].copy()
# Insult keyword filter to remove false positives (~30% noise)
INSULT_KW = r'\b(stupid|idiot|dumb|ugly|hate you|shut up|loser|pathetic|disgusting|ass|bitch|fuck|dick|moron|trash|worthless|clown|stfu|lame|suck|retard|freak|weirdo|creep|jerk|douche|scum|pig|fool)\b'
df['text_lower'] = df['tweet_text'].str.lower()
df = df[df['text_lower'].str.contains(INSULT_KW, regex=True, na=False)].copy()
df = df.drop(columns='text_lower')
df['text'] = df['tweet_text'].apply(normalize_text)
df = df[df['text'].notna() & (df['text'].str.len() > 0)]
df['label'] = 'verbal_harassment'
frames.append(df[['text', 'label']].copy())
source_tracker['kaggle_filtered'] = len(df)
print(f"  ✅ {len(df):,} rows (from ~7,823 raw other_cyberbullying)")

# ── 6. ConvAbuse verbal harassment ───────────────────────────
print("\n[6] ConvAbuse verbal harassment")
try:
    df = pd.read_csv(THREAT / 'convabuse_verbal_harassment.csv')
    df['text'] = df['text'].apply(normalize_text)
    df = df[df['text'].notna() & (df['text'].str.len() > 0)]
    df['label'] = 'verbal_harassment'
    frames.append(df[['text', 'label']].copy())
    source_tracker['convabuse_verbal'] = len(df)
    print(f"  ✅ {len(df):,} rows")
except: print("  ⚠️ File not found")

# ══════════════════════════════════════════════════════════════
# ❌ REMOVED: Davidson (noisy, double-dips)
# ❌ REMOVED: TRAC-3 verbal (Bengali/Hindi)
# ══════════════════════════════════════════════════════════════
print("\n❌ SKIPPED: TRAC-3 verbal harassment (not English)")

# ── COMBINE ──────────────────────────────────────────────────
print("\n" + "="*70)
print("COMBINING VERBAL HARASSMENT SOURCES")
print("="*70)

df_vh_all = pd.concat(frames, ignore_index=True)

# Dedup
before = len(df_vh_all)
df_vh_all['text_hash'] = df_vh_all['text'].apply(text_hash)
df_vh_all = df_vh_all.drop_duplicates(subset='text_hash', keep='first').copy()
df_vh_all = df_vh_all.drop(columns='text_hash')
print(f"  Dedup: {before:,} → {len(df_vh_all):,} ({before-len(df_vh_all):,} removed)")

# Filter min words
df_vh_all = df_vh_all[df_vh_all['text'].str.split().str.len() >= 3].copy()

# Source breakdown
print(f"\n📊 Source breakdown (pre-cap):")
for src, cnt in source_tracker.items():
    print(f"    {src:<25} {cnt:>6,} rows")
print(f"    {'TOTAL':<25} {len(df_vh_all):>6,} rows")

# Cap at 7,000 for class balance
CAP = 7_000
if len(df_vh_all) > CAP:
    df_vh_all = df_vh_all.sample(n=CAP, random_state=42)
    print(f"\n  Capped to {CAP:,} for M2 class balance")

print(f"\n  FINAL VERBAL HARASSMENT CORPUS: {len(df_vh_all):,} rows")

# Spot check
print("\n📌 Random sample (10 examples):")
for _, row in df_vh_all.sample(min(10, len(df_vh_all)), random_state=42).iterrows():
    print(f"  • {row['text'][:100]}")

In [ ]:
# ============================================================
# CELL 3 — Save Verbal Harassment Corpus
# ============================================================

print("="*70)
print("SAVING VERBAL HARASSMENT CORPUS V2")
print("="*70)

df_vh_all['language'] = 'en'
df_vh_all['source_dataset'] = 'verbal_harassment_v2'
df_vh_all['routing_method'] = 'manual'

out_path = PROCESSED / "m2_verbal_harassment_v2.csv"
df_vh_all.to_csv(out_path, index=False)

size_kb = out_path.stat().st_size / 1024
print(f"\n  ✅ Saved: {out_path.name} ({size_kb:.1f} KB)")
print(f"  ✅ Rows: {len(df_vh_all):,}")

# Validation
has_upper = df_vh_all['text'].str.contains(r'[A-Z]', regex=True).sum()
has_url   = df_vh_all['text'].str.contains(r'https?://', regex=True).sum()
print(f"\n  Validation:")
print(f"    Uppercase chars: {has_upper} (should be 0)")
print(f"    URLs remaining: {has_url} (should be 0)")

# Cross-check: no overlap with threat corpus
print(f"\n📌 Cross-check: overlap with threat corpus")
try:
    df_threat = pd.read_csv(PROCESSED / 'm2_threat_v2.csv')
    overlap = set(df_vh_all['text']) & set(df_threat['text'])
    print(f"    Overlap with threat: {len(overlap)} texts (should be 0)")
    if overlap:
        print("    ⚠️ Removing overlapping texts...")
        df_vh_all = df_vh_all[~df_vh_all['text'].isin(overlap)].copy()
        df_vh_all.to_csv(out_path, index=False)
        print(f"    ✅ After removal: {len(df_vh_all):,} rows")
except:
    print("    (threat corpus not yet built — skip cross-check)")

print(f"\n🎉 Verbal Harassment V2 complete!")